# Master catalog of games

Builds one row per game (Whizdom Game ID) from **ref_games**, joined with the other sources.
The functions are in `master_lib.py` (tests: `tests/test_master_lib.py`).

**Keys**
- Game: name + provider. The provider is the studio that made the game, not the platform.
- `whizdom_game_id = uuid5(NAMESPACE, "<provider>/<game>")`: the same game always gets the same ID.
- `provider_id = uuid5(NAMESPACE, "<provider>")`.

**Cell types**
- **MANUAL**: reads ClickHouse or S3. Only run them by hand. Their flags are off by default.
- Everything else works with local files.

**Output**: `output/master/master_catalog.xlsx`.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

HERE = Path.cwd() if (Path.cwd() / "master_lib.py").exists() else Path.cwd() / "catalog_project"
sys.path[:0] = [str(HERE), str(HERE.parent)]
import master_lib as ml  # noqa: E402
import slotcatalog_json  # noqa: E402
from priority_list import by_key, key  # noqa: E402

config = yaml.safe_load((HERE / "config.yaml").read_text())
OUT = HERE / config["master_out"]
OUT.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 200)
print("ID version", ml.ID_VERSION, "| namespace", ml.NAMESPACE)

ID version 1 | namespace 907a5265-b739-542f-8906-b9b0c91432a8


## 1. Base list (ref_games)

- Every column of ref_games, seen since `master_seen_since` (`config.yaml`).
- Run the MANUAL cell once to export `output/master/ref_games.parquet`.

In [2]:
# MANUAL (ClickHouse, read only): set to True to export a fresh ref_games
REFRESH_REF_GAMES = False
if REFRESH_REF_GAMES:
    from active_games import query
    from signal_audit.data.clickhouse_mirror import client
    query(client(), "ref_games_master").to_parquet(OUT / "ref_games.parquet", index=False)

In [3]:
path = OUT / "ref_games.parquet"
assert path.exists(), "ref_games.parquet is missing: run the MANUAL cell above once (REFRESH_REF_GAMES = True)"
ref = pd.read_parquet(path)
assert ref["game_uid"].is_unique, "ref_games must have one row per game_uid"
REF_DATE = pd.Timestamp(path.stat().st_mtime, unit="s").date()

games = ref[ref["last_seen"] >= pd.Timestamp(config["master_seen_since"])].add_prefix("ref_").reset_index(drop=True)
print(f"ref_games of {REF_DATE}: {len(ref)} games | base list: {len(games)} seen since {config['master_seen_since']}")

ref_games of 2026-10-30: 20078 games | base list: 18100 seen since 2025-01-01


## 2. Other sources

- Client Area, Slot Catalog URL file, games_enriched and the raw Slot Catalog JSON.
- The MANUAL cell downloads the JSON from S3 again, only if needed.

In [4]:
# MANUAL (S3, profile in config.yaml): set to True to download and unpack the raw JSON again
REFRESH_RAW = False
if REFRESH_RAW:
    import io, zipfile, boto3
    bucket, zip_key = config["sa_raw_zip"].removeprefix("s3://").split("/", 1)
    body = boto3.Session(profile_name=config["sa_raw_profile"]).client("s3").get_object(Bucket=bucket, Key=zip_key)["Body"].read()
    target = HERE / config["slotcatalog_json"]
    target.mkdir(exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(body)) as z:
        for n in z.namelist():
            if n.startswith("data/raw/") and n.endswith(".json"):
                (target / Path(n).name).write_bytes(z.read(n))

In [5]:
ca = pd.read_csv(HERE / config["client_area"], dtype={"Core Id Desktop": str, "Core Id Mobile": str})
sa = pd.read_excel(HERE / config["sa_urls"], dtype=str).sort_values("sc_url", na_position="last")
# the index and the ranking of the old priority file of SA are not data of the game
enriched = pd.read_excel(HERE / config["games_enriched"], dtype=str).drop(columns=["Unnamed: 0", "dervied_rank", "total_bets_pct", "totatl_bets_cumsum"])
info = slotcatalog_json.load(HERE / config["slotcatalog_json"])
raw = ml.load_raw(HERE / config["slotcatalog_json"])
print(f"Client Area {len(ca)} rows | Slot Catalog file {len(sa)} rows | games_enriched {len(enriched)} rows | raw JSON {len(raw)} files")

Client Area 16273 rows | Slot Catalog file 15855 rows | games_enriched 1071 rows | raw JSON 9532 files


## 3. Join the sources

- Client Area, SA file and games_enriched join by game id and platform.
- The JSON joins by the UUID of the SA file.
- Studio: ref_games, then Client Area, then JSON, then platform (`studio_source`).

In [6]:
games["key"] = key(games["ref_game_id"], games["ref_provider_code"])
ca_all = by_key(ca, ["Core Id Desktop", "Core Id Mobile"], "Platform", list(ca.columns)).rename(columns=lambda c: "ca_" + ml.snake(c))
sa_all = by_key(sa, ["game_src_id", "core_id_desktop"], "platform", list(sa.columns)).add_prefix("sa_")
ge_all = by_key(enriched, ["game_src_id", "core_desktop_id", "core_mobile_id"], "platform_code", list(enriched.columns)).add_prefix("ge_")
games = games.join(ca_all, on="key").join(sa_all, on="key").join(ge_all, on="key")

games["json_studio"] = games["sa_game_id"].map(info.set_index("sc_uuid")["sc_provider"])
games["studio"], games["studio_source"] = ml.first_labelled(
    games, {"ref_studios": "ref_games", "ca_provider": "client_area", "json_studio": "slotcatalog_json", "ref_provider_name": "platform"})
games["studio_source"].value_counts()

studio_source
client_area         13273
platform             3119
ref_games            1411
none                  201
slotcatalog_json       96
Name: count, dtype: int64

## 4. Provider names

- `providers.yaml` maps each spelling to one standard provider.
- `provider_check = true`: the provider has a look-alike name not merged yet. A person decides.

In [7]:
aliases = ml.load_providers()
all_names = pd.concat([ca["Provider"], sa["provider"], info["sc_provider"], games["studio"]])
candidates = ml.provider_candidates(all_names, aliases)      # look-alike names not merged yet

games["provider_slug"], _ = ml.canonical_provider(games["studio"], aliases)
candidates

,slug,rows,core,standard
0,high5,243,high5,high5
1,high-5-games,84,high5,high-5-games


## 5. Name, studio and IDs

**Name**: ref_games first, then Client Area, SA file or games_enriched (`name_source`).
- `fix_possessive` restores apostrophes (`Santa S Gifts` becomes `Santa's Gifts`).
- Sports betting has no name in any source: its name comes from `explicit_names` in `config.yaml`.
- No name at all: no ID, the game goes to the sheet `no name`.

**Studio**: the platform-only studio is replaced by the studio of the same game name in a better source, only if the RTP matches (`same_name`).

In [8]:
games["game_name"], games["name_source"] = ml.first_labelled(
    games, {"ref_game_name": "ref_games", "ca_game_name": "client_area", "sa_game_name": "slotcatalog_file", "ge_game_name": "games_enriched"})

explicit = (games["ref_provider_code"] + "/" + games["ref_game_id"].astype(str)).map(config["explicit_names"])
use_explicit = games["game_name"].isna() & explicit.notna()          # only where no source has a name
games.loc[use_explicit, "game_name"], games.loc[use_explicit, "name_source"] = explicit[use_explicit], "explicit"

games["game_name"] = games["game_name"].map(ml.fix_possessive)
games["game_slug"] = games["game_name"].map(ml.slugify)
games["name_source"].value_counts()

name_source
ref_games           15915
none                 1804
client_area           247
slotcatalog_file      110
games_enriched         18
explicit                6
Name: count, dtype: int64

In [9]:
games["rtp"], _ = ml.first_of(games, ["ref_rtp", "ca_rtp"])
games["rtp"] = pd.to_numeric(games["rtp"], errors="coerce")

same_name = ml.studio_from_same_name(games)
use = same_name["provider_slug"].notna()
games.loc[use, ["provider_slug", "studio"]] = same_name.loc[use, ["provider_slug", "studio"]]
games.loc[use, "studio_source"] = "same_name"
print(use.sum(), "games take the studio of the same game name")

167 games take the studio of the same game name


In [10]:
games["whizdom_game_id"] = [ml.whizdom_id(g, p) for g, p in zip(games["game_slug"], games["provider_slug"])]
games["provider_id"] = games["provider_slug"].map(ml.provider_id)
print(games["whizdom_game_id"].notna().sum(), "games with an ID,", games["whizdom_game_id"].nunique(), "distinct IDs |",
      games["whizdom_game_id"].isna().sum(), "with no name, so no ID |", games["provider_id"].nunique(), "providers")

16291 games with an ID, 16005 distinct IDs | 1809 with no name, so no ID | 160 providers


## 6. Match with the raw JSON

- `exact`: the JSON found by platform ids has the same name and provider.
- `review`: only one way finds the JSON, they disagree, or a close name exists (similarity 0.9 or more).
- `no_match`: nothing found.

`sc_` columns come from the JSON found by ids, or else by name and provider.

In [11]:
js = info[["sc_uuid", "sc_provider"]].copy()
js["name"] = js["sc_uuid"].map(sa.drop_duplicates("game_id").set_index("game_id")["game_name"])
js["provider_slug"], _ = ml.canonical_provider(js["sc_provider"], aliases)
js["whizdom_game_id"] = [ml.whizdom_id(ml.slugify(n), p) for n, p in zip(js["name"], js["provider_slug"])]
js["game_slug"] = js["name"].map(ml.slugify)
js_with_id = js.dropna(subset=["whizdom_game_id"])

games["json_by_name"] = games["whizdom_game_id"].map(js_with_id.drop_duplicates("whizdom_game_id").set_index("whizdom_game_id")["sc_uuid"])
games["json_by_id"] = games["sa_game_id"].where(games["sa_game_id"].isin(set(js["sc_uuid"])))
games["json_by_id_whizdom"] = games["json_by_id"].map(js.set_index("sc_uuid")["whizdom_game_id"])

games["close_name"] = None
no_hit = games["json_by_name"].isna() & games["json_by_id"].isna()
games.loc[no_hit, "close_name"] = ml.close_names(games[no_hit], js_with_id)

games["match_status"] = ml.match_status(games["whizdom_game_id"], games["json_by_name"], games["json_by_id_whizdom"], games["close_name"])
games["sc_uuid"] = games["json_by_id"].fillna(games["json_by_name"])
games = games.join(info.set_index("sc_uuid"), on="sc_uuid")
games["match_status"].value_counts()

match_status
no_match    9164
exact       7082
review      1854
Name: count, dtype: int64

## 7. URL, status and flags

- `url`: SA file, then JSON, then Client Area.
- `status` and `game_type`: ref_games, then Client Area.
- `same_platform_conflict`: two `game_uid` on the same platform share the ID (same game, same RTP).

In [12]:
ca_url = games["ca_external_resource"].where(games["ca_external_resource"].astype(str).str.contains("slotcatalog"))
games["json_url"] = games["sc_uuid"].map(raw.set_index("source_id")["source_url"])
games["url"], games["url_source"] = ml.first_labelled(
    games.assign(ca_url=ca_url), {"sa_sc_url": "slotcatalog_file", "json_url": "raw_json", "ca_url": "client_area"})
games["no_url"] = games["url"].isna()

games["game_type"] = games["ref_game_type"].fillna(games["ca_game_type"])
games["status"], _ = ml.first_of(games, ["ref_status", "ca_status"])
games["is_sports"] = games["ref_provider_code"].isin(config["sports_provider_codes"])
games["provider_check"] = games["provider_slug"].isin(set(candidates["standard"]))

missing = pd.DataFrame({"name": games["game_name"].isna(), "studio": games["studio_source"].isin(["platform", "none"]),
                        "game_type": games["game_type"].isna()})
games["missing_data"] = missing.apply(lambda row: ", ".join(row.index[row]), axis=1)
games["id_shared"] = games.groupby("whizdom_game_id")["ref_game_uid"].transform("count").gt(1)
games["same_platform_conflict"] = games["whizdom_game_id"].notna() & games.duplicated(["whizdom_game_id", "ref_provider_code"], keep=False)

## 8. Catalog

One row per ID: the best match of its `game_uid`. Order of rows, same as `active_games.py`:
GGR (highest first), rounds, then `game_uid`. The first rows are the 99 % list.
The games with no GGR go after, by provider and name.

Before writing, the IDs are compared with the last run. A changed ID means a rule changed.

In [13]:
identities = games[["ref_game_uid", "ref_game_id", "ref_provider_code", "ref_provider_name", "game_name", "name_source", "whizdom_game_id",
                    "provider_id", "studio_source", "match_status"]].rename(columns=lambda c: c.removeprefix("ref_"))
changes = ml.compare_ids(identities, OUT / "game_identities.parquet")
changes["change"].value_counts().to_dict() or "no changes against the last run"

{'changed': 6}

In [14]:
activity = pd.read_csv(HERE / config["master_ggr"], usecols=["game_uid", "ggr_eur_6m", "rounds_6m"], dtype={"game_uid": str})
ACTIVITY_DATE = pd.Timestamp((HERE / config["master_ggr"]).stat().st_mtime, unit="s").date()
games = games.join(activity.set_index("game_uid"), on="ref_game_uid")

# the best row of each ID: exact first, then review, then no_match; with URL first
games["order"] = games["match_status"].map({"exact": 0, "review": 1, "no_match": 2})
best = games.dropna(subset=["whizdom_game_id"]).sort_values(["whizdom_game_id", "order", "no_url"])
by_id = best.groupby("whizdom_game_id")

master = best.drop_duplicates("whizdom_game_id").copy()
master["provider"] = master["provider_slug"]
master["id_version"] = ml.ID_VERSION
master["platforms"] = master["whizdom_game_id"].map(by_id["ref_provider_code"].agg(lambda s: ", ".join(sorted(set(s.dropna())))))
master["platform_rows"] = master["whizdom_game_id"].map(by_id.size())
for column in ["ggr_eur_6m", "rounds_6m"]:                     # the sum of all the game_uid of the game
    master[column] = master["whizdom_game_id"].map(by_id[column].sum(min_count=1))
master["order_uid"] = master["whizdom_game_id"].map(by_id["ref_game_uid"].min()).where(master["ggr_eur_6m"].notna())

master = master.sort_values(["ggr_eur_6m", "rounds_6m", "order_uid", "provider", "game_name", "whizdom_game_id"],
                            ascending=[False, False, True, True, True, True], na_position="last").reset_index(drop=True)
with_ggr = master["ggr_eur_6m"].notna()
assert with_ggr.iloc[:with_ggr.sum()].all() and master["ggr_eur_6m"].dropna().is_monotonic_decreasing, "games with GGR must come first, by GGR"
print(f"{len(master)} IDs, {with_ggr.sum()} with GGR: {master['ggr_eur_6m'].sum():,.0f} of {activity['ggr_eur_6m'].sum():,.0f} EUR of the list")

16005 IDs, 3489 with GGR: 246,791,398 of 250,086,908 EUR of the list


In [15]:
OWN = ["whizdom_game_id", "game_name", "ggr_eur_6m", "rounds_6m", "name_source", "provider", "provider_id", "id_version", "studio",
       "studio_source", "status", "game_type", "is_sports", "rtp", "url", "url_source", "match_status", "close_name", "missing_data",
       "no_url", "provider_check", "id_shared", "same_platform_conflict", "platforms", "platform_rows"]
REPEATED = ["ref_game_name", "ref_status", "ref_game_type", "ref_rtp", "ca_game_type", "ref_provider_name"]   # same as game_name, status...
source_columns = [c for c in games if c.startswith(tuple(ml.SOURCES))]
left_out = ml.platform_id_columns(source_columns) + REPEATED

master = master[OWN + [c for c in source_columns if c not in left_out]]
empty = [c for c in master if c not in OWN and master[c].isna().all()]
master = master.drop(columns=empty)
print(master.shape[1], "columns | left out: ids of platforms", ml.platform_id_columns(source_columns), "| repeated", REPEATED, "| empty", empty)

119 columns | left out: ids of platforms ['ref_game_uid', 'ref_game_id', 'ca_core_id_desktop', 'ca_core_id_mobile', 'ca_core_launch_id_desktop', 'ca_core_launch_id_mobile', 'ca_start_id_desktop', 'ca_start_id_mobile', 'sa_game_id', 'sa_game_src_id', 'sa_id', 'sa_core_id_desktop', 'sa_game_platform_id', 'ge_game_id', 'ge_game_src_id', 'ge_external_game_id', 'ge_catalog_id', 'ge_core_desktop_id', 'ge_core_mobile_id', 'ge_game_record_seq', 'sc_uuid'] | repeated ['ref_game_name', 'ref_status', 'ref_game_type', 'ref_rtp', 'ca_game_type', 'ref_provider_name'] | empty ['ge_theme_x', 'ge_tags', 'ge_hit_frequency', 'ge_grid_layout', 'ge_max_win_multiplier', 'ge_release_date', 'ge_supported_currencies', 'ge_is_featured', 'ge_is_new', 'ge_is_exclusive', 'ge_display_order', 'ge_description', 'ge_game_features_text', 'ge_thumbnail_url', 'ge_demo_url', 'ge_status']


## 9. Output

- `master_catalog.xlsx`: the catalog, how each column is made (`columns`), where the data come from with their queries (`sources`), and the lists to review.
- `master_games.parquet`: the same catalog.
- `game_identities.parquet`: each `game_uid` and its ID, to join the activity.
- `raw_sources.parquet`: the full raw JSON.

In [16]:
raw_sources = raw.merge(js[["sc_uuid", "whizdom_game_id"]], left_on="source_id", right_on="sc_uuid", how="left").drop(columns="sc_uuid")
raw_sources.insert(0, "source", "slotcatalog_json")

master.to_parquet(OUT / "master_games.parquet", index=False)
identities.to_parquet(OUT / "game_identities.parquet", index=False)
raw_sources.to_parquet(OUT / "raw_sources.parquet", index=False)

In [17]:
flags = pd.Series({"IDs in the catalog": len(master), "exact": master["match_status"].eq("exact").sum(),
                   "review": master["match_status"].eq("review").sum(), "no_match": master["match_status"].eq("no_match").sum(),
                   "no_url": master["no_url"].sum(), "missing_data": master["missing_data"].ne("").sum(),
                   "provider_check": master["provider_check"].sum(), "id_shared": master["id_shared"].sum(),
                   "same_platform_conflict": master["same_platform_conflict"].sum()}, name="IDs of the catalog").astype(int)

providers = games.dropna(subset=["provider_id"]).groupby(["provider_id", "provider_slug"]).agg(
    names=("studio", lambda s: ", ".join(sorted(set(s.dropna())))), games=("whizdom_game_id", "nunique"),
    provider_check=("provider_check", "any")).reset_index().rename(columns={"provider_slug": "provider"})
providers.insert(4, "in_providers_yaml", providers["provider"].isin(set(aliases.values())))
providers = providers.sort_values("games", ascending=False)

review = games[games["match_status"].eq("review")][["whizdom_game_id", "game_name", "studio", "provider_slug", "ref_provider_code", "close_name", "url"]]
no_name = games[games["whizdom_game_id"].isna()][["ref_provider_code", "ref_provider_name", "ref_game_id", "studio", "game_type", "url", "ggr_eur_6m", "rounds_6m"]]
no_name = no_name.sort_values(["ggr_eur_6m", "rounds_6m"], ascending=False, na_position="last")      # most GGR first

notes = {**{c: {"meaning": v["meaning"]} for c, v in yaml.safe_load((HERE / "columns.yaml").read_text()).items()},
         **yaml.safe_load((HERE / "master_columns.yaml").read_text())}
originals = {**{"ref_" + c: c for c in ref.columns}, **{"ca_" + ml.snake(c): c for c in ca.columns}, **{"sa_" + c: c for c in sa.columns},
             **{"ge_" + c: c for c in enriched.columns}, **{v: "raw_data." + k for k, v in slotcatalog_json.FIELDS.items()}}
sources = ml.source_notes(HERE / "master_sources.yaml", HERE, {
    "ref_": len(ref), "ggr_eur_6m, rounds_6m": len(activity), "ca_": len(ca), "sa_": len(sa), "ge_": len(enriched), "sc_": len(raw),
    "explicit_names": len(config["explicit_names"]), "providers.yaml": len(set(aliases.values()))})
facts = {
    "built on": str(pd.Timestamp.today().date()),
    "ref_games.parquet exported on": str(REF_DATE),
    "id version": ml.ID_VERSION,
    "games of ref_games": len(ref),
    f"base list: games seen since {config['master_seen_since']}": len(games),
    "games with no name (no Whizdom Game ID, sheet no name)": len(no_name),
    "names taken from the Client Area, SA file or games_enriched": int(games["name_source"].isin(["client_area", "slotcatalog_file", "games_enriched"]).sum()),
    "names written on purpose (sports betting, explicit_names in config.yaml)": int(games["name_source"].eq("explicit").sum()),
    "studios taken from the same game name on another platform": int(games["studio_source"].eq("same_name").sum()),
    "columns left out because they are empty in every row": len(empty),
    "Whizdom Game IDs (rows of catalog)": len(master),
    "Provider IDs (sheet providers)": len(providers),
    "Whizdom Game IDs changed against the last run": int(changes["change"].eq("changed").sum()),
    "raw Slot Catalog JSON kept (raw_sources.parquet)": len(raw_sources),
    "raw JSON of games not in the base list": int((~raw_sources["whizdom_game_id"].isin(set(games["whizdom_game_id"]))).sum()),
    "GGR file (by game_uid)": f"{config['master_ggr']}, built on {ACTIVITY_DATE}",
    "games of the catalog with GGR (the first rows)": int(with_ggr.sum()),
    "GGR of the games with no name, so not in the catalog (EUR)": round(float(games.loc[games["whizdom_game_id"].isna(), "ggr_eur_6m"].sum())),
    "column prefixes": "ref_ ref_games, ca_ Client Area, sa_ Slot Catalog URL file, sc_ raw Slot Catalog JSON, ge_ games_enriched; no prefix: made here",
    "order of the rows": "same as active_games.py: GGR (highest first), rounds (highest first), game_uid; the first rows are the 99% list. Games with no GGR go after, by provider and name",
    "one row per ID": "a game on several platforms shows the row with the best match",
    "sheets": "catalog: one row per game | columns: source, origin, how it is made and meaning of each column | sources: files, queries and joins | providers | flags | review | no name",
    "old ids": "ids of platforms and brands are not in the catalog; game_identities.parquet links each game_uid to its Whizdom Game ID",
}
read_me = pd.DataFrame(list(facts.items()), columns=["item", "value"])

In [18]:
ml.write_excel(OUT / "master_catalog.xlsx", {
    "read me": read_me, "catalog": master, "columns": ml.column_notes(master, notes, originals), "sources": sources,
    "providers": providers, "flags": flags.rename_axis("flag").reset_index(), "review": review, "no name": no_name},
    freeze={"catalog": "D2"}, links={"catalog": "url", "review": "url", "no name": "url"},
    wide={"columns": {"source": 30, "origin": 40, "how": 90, "meaning": 55, "example": 30},
          "sources": {"source": 30, "where": 40, "how": 70, "joined": 60, "query": 90}})
print(f"catalog {len(master)} games x {master.shape[1]} columns | providers {len(providers)} | no name {len(no_name)} -> {OUT / 'master_catalog.xlsx'}")
flags

catalog 16005 games x 119 columns | providers 160 | no name 1809 -> /home/javier/whizdom-churn/catalog_project/output/master/master_catalog.xlsx


IDs in the catalog        16005
exact                      7046
review                     1704
no_match                   7255
no_url                     5634
missing_data               1466
provider_check               73
id_shared                   281
same_platform_conflict      212
Name: IDs of the catalog, dtype: int64

## 10. Navid's final file (pending)

Set `navid_final` in `config.yaml` when it arrives. Only the URLs and metadata join, not the reviews or full texts of third-party sites.

In [19]:
if not config.get("navid_final"):
    print("Pending: the final file of Navid and the schema. Set navid_final in config.yaml when they arrive.")

Pending: the final file of Navid and the schema. Set navid_final in config.yaml when they arrive.
